#Raw Citations

In [ ]:
import os
import json
import itertools
import csv
from google.colab import drive

drive.mount('/content/drive')

# --- CONFIGURATION ---
DATA_DIRS = [
    "/content/drive/MyDrive/NLP Data/Rayane_Preprocessing"
]

OUTPUT_DIR = "/content/drive/MyDrive/NLP Data/Rayane_Preprocessing/Co-Citations/"
CHECKPOINT_FILE = "/content/drive/MyDrive/NLP Data/Rayane_Preprocessing/Co-Citations/processed_files.txt"
FILES_PER_BATCH = 50
# ---------------------

os.makedirs(OUTPUT_DIR, exist_ok=True)


# -------- CHECKPOINT LOGIC (SAFE) --------
def load_checkpoint(path):
    """Load processed files from previous runs."""
    if not os.path.exists(path):
        return set()
    with open(path, "r") as f:
        return set(f.read().splitlines())


def save_checkpoint(path, processed_ids):
    """Overwrite checkpoint with ALL processed file IDs after each batch."""
    with open(path, "w") as f:
        for uid in processed_ids:
            f.write(uid + "\n")
# ------------------------------------------


# -------- FILE COLLECTION ----------
def collect_all_json_files(dirs):
    all_files = []
    for i, d in enumerate(dirs):
        if not os.path.exists(d):
            print(f"WARNING: Directory missing: {d}")
            continue

        print(f"Scanning: {d}")
        files = [f for f in os.listdir(d) if f.endswith(".json")]
        for f in files:
            all_files.append((os.path.join(d, f), f"dir{i}_{f}"))

        print(f"  → {len(files)} JSON files")
    return all_files
# ------------------------------------


print("="*70)
print("      EXTRACTING CO-CITATIONS (CRASH-SAFE VERSION)")
print("="*70)

# Load file list
all_files = collect_all_json_files(DATA_DIRS)
if not all_files:
    print("NO JSON FILES FOUND. EXITING.")
    exit()

# Load resume state
processed = load_checkpoint(CHECKPOINT_FILE)
remaining = [(p, uid) for p, uid in all_files if uid not in processed]

print(f"Total files: {len(all_files)}")
print(f"Already processed: {len(processed)}")
print(f"Remaining: {len(remaining)}")
print("-"*70)

if not remaining:
    print("ALL FILES PROCESSED!")
    exit()

# -------- PROCESSING LOOP (BATCH SAFE) --------
header = ["paper1", "paper2", "citing_article"]
current_batch_index = (len(processed) // FILES_PER_BATCH) + 1

batch_files = []  # store file IDs of CURRENT BATCH

for i, (file_path, uid) in enumerate(remaining, 1):

    # Start new batch
    if len(batch_files) == 0:
        batch_filename = f"co_citations_batch_{current_batch_index:04d}.csv"
        batch_path = os.path.join(OUTPUT_DIR, batch_filename)

        print(f"\n📁 Starting Batch {current_batch_index}: {batch_filename}")

        outfile = open(batch_path, "w", newline="", encoding="utf-8")
        writer = csv.writer(outfile)
        writer.writerow(header)

    # PROCESS SINGLE FILE
    filename = os.path.basename(file_path)
    print(f"[{i}/{len(remaining)}] {filename}")

    try:
        with open(file_path, "r", encoding="utf-8") as f:
            articles = json.load(f)
    except Exception as e:
        print(f"  ❌ Error reading: {e}")
        continue

    if isinstance(articles, dict):
        articles = [articles]
    elif not isinstance(articles, list):
        print("  ❌ Unexpected file format")
        continue

    rows = []
    for art in articles:
        article_id = art.get("id", "").replace("https://openalex.org/", "")
        refs = art.get("referenced_works", [])

        if not isinstance(refs, list) or len(refs) < 2:
            continue

        refs = [
            r.replace("https://openalex.org/", "")
            for r in refs if isinstance(r, str) and r.startswith("https://openalex.org/")
        ]
        if len(refs) < 2:
            continue

        for p1, p2 in itertools.combinations(refs, 2):
            rows.append([p1, p2, article_id])

    if rows:
        writer.writerows(rows)
        outfile.flush()
        os.fsync(outfile.fileno())  # full safety
        print(f"  ✓ {len(rows)} co-citations")

    batch_files.append(uid)

    # If batch full → close + checkpoint
    if len(batch_files) >= FILES_PER_BATCH:
        outfile.close()
        print(f"✓ Batch {current_batch_index} completed.")

        # Add batch to processed (SAFELY)
        processed.update(batch_files)
        save_checkpoint(CHECKPOINT_FILE, processed)

        # Reset batch
        batch_files = []
        current_batch_index += 1

        print(f"Checkpoint updated.")

# -------- SAVE LAST BATCH IF NOT EMPTY --------
if batch_files:
    outfile.close()
    print(f"✓ Final Batch {current_batch_index} completed.")

    processed.update(batch_files)
    save_checkpoint(CHECKPOINT_FILE, processed)
    print("Checkpoint updated.")

print("\n" + "="*70)
print("      ALL BATCHES SAVED SUCCESSFULLY")
print("="*70)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
      EXTRACTING CO-CITATIONS (CRASH-SAFE VERSION)
Scanning: /content/drive/MyDrive/NLP Data/Rayane_Preprocessing
  → 1417 JSON files
Total files: 1417
Already processed: 250
Remaining: 1167
----------------------------------------------------------------------

📁 Starting Batch 6: co_citations_batch_0006.csv
[1/1167] Social Work Education and Practice.json
  ✓ 2997748 co-citations
[2/1167] Evaluation of Teaching Practices.json
  ✓ 2557602 co-citations
[3/1167] Regional Economics and Spatial Analysis.json
  ✓ 11543658 co-citations
[4/1167] Personality Traits and Psychology.json
  ✓ 11210171 co-citations
[5/1167] Global trade and economics.json
  ✓ 8805702 co-citations
[6/1167] Information Systems Theories and Implementation.json
  ✓ 8298295 co-citations
[7/1167] Corruption and Economic Development.json
  ✓ 3133089 co-citations
[8/1167] Merger and Competition 

#Citations_Count

In [ ]:
import os
import json
import itertools
import csv
import pandas as pd
from google.colab import drive

# --- CONFIGURATION (Adjust Paths as needed) ---
DATA_DIR = "/content/drive/MyDrive/NLP Data/Rayane_Preprocessing"
# CSV where all raw pairs are written incrementally (for article-level safety)
CHECKPOINT_OUTPUT = "/content/drive/MyDrive/NLP Data/Rayane_Preprocessing/co_citations_checkpoint.csv"
# Text file to track which JSON files have been fully processed (for file-level resumability)
PROGRESS_LOG = "/content/drive/MyDrive/NLP Data/Rayane_Preprocessing/progress_log.txt"
# Final output file for the aggregated counts
COUNT_OUTPUT = "/content/drive/MyDrive/NLP Data/Rayane_Preprocessing/co_citation_counts.csv"
# -----------------------------------------------

# Mount Drive (if not already done)
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount('/content/drive')

print("--- 1/2: EXTRACTING RAW CO-CITATIONS WITH ROBUST CHECKPOINTING ---")

# 1. LOAD COMPLETED FILES LIST for resumability
completed_files = set()
if os.path.exists(PROGRESS_LOG):
    with open(PROGRESS_LOG, 'r') as f:
        completed_files = set(f.read().splitlines())
    print(f"Loaded {len(completed_files)} files from progress log. Resuming work.")
else:
    print("No progress log found. Starting from the beginning.")

# 2. INITIALIZE CHECKPOINT CSV
write_header = not os.path.exists(CHECKPOINT_OUTPUT)

# Open both the checkpoint CSV and the progress log for appending
with open(CHECKPOINT_OUTPUT, "a", newline="", encoding="utf-8") as outfile, \
     open(PROGRESS_LOG, "a") as logfile:

    writer = csv.writer(outfile)
    if write_header:
        writer.writerow(["paper1", "paper2", "citing_article"])

    files_to_process = os.listdir(DATA_DIR)

    for filename in files_to_process:
        if not filename.endswith(".json"):
            continue

        # --- RESUMABILITY CHECK (Skip entire file if already logged) ---
        if filename in completed_files:
            continue

        file_path = os.path.join(DATA_DIR, filename)
        print(f"Processing: {filename}")

        # --- START FILE PROCESSING ---
        article_rows = []
        try:
            with open(file_path, "r", encoding="utf-8") as f:
                articles = json.load(f)
        except Exception as e:
            print(f"Error reading or parsing {filename} → {e}. Skipping file.")
            continue # Skip to next file if parsing fails

        if isinstance(articles, dict):
            articles = [articles]
        elif not isinstance(articles, list):
            print(f"Unexpected format in {filename}. Skipping file.")
            continue

        for art in articles:
            article_id = art.get("id", "").replace("https://openalex.org/", "")
            referenced_works = art.get("referenced_works", [])

            if not isinstance(referenced_works, list) or len(referenced_works) < 2:
                continue

            referenced_clean = [
                ref.replace("https://openalex.org/", "")
                for ref in referenced_works
                if isinstance(ref, str) and ref.startswith("https://openalex.org/")
            ]

            # Deduplication Fix: Ensures unique references per article
            referenced_clean = list(set(referenced_clean))

            if len(referenced_clean) < 2:
                continue

            # Generate and normalize pairs, writing to disk immediately
            for p1, p2 in itertools.combinations(referenced_clean, 2):
                # Normalization Fix: Ensure consistent order (p1 < p2)
                normalized_pair = tuple(sorted((p1, p2)))

                # --- ARTICLE-LEVEL CHECKPOINTING ---
                # Write the pair immediately to disk. If crash occurs here,
                # only this single article's pairs might be lost, not the whole file.
                writer.writerow([normalized_pair[0], normalized_pair[1], article_id])

        # --- FILE-LEVEL CHECKPOINTING ---
        # Mark file as complete only after successfully processing all articles
        logfile.write(filename + '\n')
        print(f"-> Successfully Checkpointed: {filename}")

print(f"\n✔ RAW CO-CITATIONS CHECKPOINTED TO {CHECKPOINT_OUTPUT}")

# --- 2/2: AGGREGATE COUNTS FROM CHECKPOINT FILE ---
print("\n--- 2/2: CALCULATING FINAL CO-CITATION COUNTS ---")

try:
    # Load the checkpoint file from disk
    # The 'paper1' and 'paper2' columns are already normalized (sorted)
    df_pairs = pd.read_csv(CHECKPOINT_OUTPUT)

    # Group by the pair and count the occurrences (vectorized and fast)
    co_citation_counts = df_pairs.groupby(['paper1', 'paper2']).size().reset_index(name='count')

    # Save the final results
    co_citation_counts.to_csv(COUNT_OUTPUT, index=False)

    print(f"\n✔ FINAL CO-CITATION COUNTS SAVED TO {COUNT_OUTPUT}")
    print(f"   Total unique co-citation pairs found: {len(co_citation_counts)}")

except Exception as e:
    print(f"An error occurred during final aggregation. Check if the checkpoint file is corrupted: {e}")